# Institutions and the Intergenerational Sustainability Dilemma
### Decomposing *how* mechanisms work: level, coordination, and the self–other gap

**Game:** one-person Intergenerational Goods Game (IGG, after Hauser et al. 2014). Each
participant is a member of a sequence and chooses how many of 100 points to extract.
**Taking ≤ 50 is the sustainable choice** (the common account refills to 100 for the next
member); taking > 50 collapses it.

**Five conditions** (`Treatment`): 1 = Baseline IGG, 2 = IA (accountability),
3 = FAB (future-ahead-and-back), 4 = SDR (self-discrepancy reflection),
5 = endogenous Selection.

**Research aspect.** An institution can raise sustainability through three distinct channels,
and this design lets us separate them:

1. **Level** — does extraction fall / does the sustainable share rise?
2. **Coordination** — does behaviour compress onto a sustainable *focal point* (dispersion ↓)?
3. **Self–other / self–ideal discrepancy** — FAB records what you *demand of predecessors*
   vs. what you *take yourself*; SDR records *actual / ideal / ought* selves vs. the final
   choice. We can measure the moral gap **and** whether it translated into behaviour.

> Outcome variable throughout: `final_amount` (0–100). Lower = more sustainable.


In [ ]:
# === 0. Environment =========================================================
!pip -q install pyreadstat statsmodels seaborn scipy --upgrade 2>/dev/null
import numpy as np, pandas as pd
import scipy.stats as ss
import statsmodels.api as sm
import matplotlib.pyplot as plt, seaborn as sns
from itertools import combinations
sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.width", 140, "display.max_columns", 40)
print("ready")

## 1. Load the data
Run this cell and pick `Final_data_for_analysis_stata.dta` from your machine.
If you've put the file in your Drive or the Colab working directory, just set `PATH` instead.

In [ ]:
# === 1. Load ===============================================================
PATH = None  # e.g. "Final_data_for_analysis_stata.dta" if already uploaded

if PATH is None:
    try:
        from google.colab import files
        up = files.upload()                       # choose the .dta
        PATH = list(up.keys())[0]
    except Exception:
        PATH = "Final_data_for_analysis_stata.dta"

try:
    import pyreadstat
    df, meta = pyreadstat.read_dta(PATH)
except Exception:
    df = pd.read_stata(PATH)

TMAP = {1:"Baseline", 2:"IA", 3:"FAB", 4:"SDR", 5:"Selection"}
ORDER = ["Baseline", "IA", "FAB", "SDR", "Selection"]
df["Mech"] = df["Treatment"].map(TMAP)
df["sustainable"] = (df["final_amount"] <= 50).astype(int)   # refill threshold
print(df.shape)
df[["section","Treatment","Mech","final_amount","sustainable"]].head()

## 2. Channel 1 — LEVEL effect
Does the mechanism change *how much* people take? Because the outcome piles up at the
focal values 50 and 100, we lean on rank-based and proportion tests, not t-tests.
This mirrors Table 1 of the 2019 paper but with a continuous outcome.

In [ ]:
# === 2a. Descriptive table =================================================
desc = (df.groupby("Mech")
          .agg(N=("final_amount","size"),
               mean=("final_amount","mean"),
               median=("final_amount","median"),
               IQR=("final_amount", lambda x: x.quantile(.75)-x.quantile(.25)),
               sd=("final_amount","std"),
               pct_sustainable=("sustainable", lambda x: 100*x.mean()))
          .reindex(ORDER).round(2))
desc

In [ ]:
# === 2b. Omnibus + pairwise tests vs Baseline ==============================
groups = {m: df.loc[df.Mech==m, "final_amount"].dropna() for m in ORDER}

H, p = ss.kruskal(*groups.values())
print(f"Kruskal-Wallis across all 5 conditions:  H={H:.2f}, p={p:.4f}\n")

print("Pairwise vs Baseline  (Mann-Whitney on amount, chi-square on sustainable dummy):")
base = groups["Baseline"]
base_sus = df.loc[df.Mech=="Baseline","sustainable"]
rows=[]
for m in ["IA","FAB","SDR","Selection"]:
    u,pu = ss.mannwhitneyu(base, groups[m], alternative="two-sided")
    ct = pd.crosstab(df.loc[df.Mech.isin(["Baseline",m]),"Mech"],
                     df.loc[df.Mech.isin(["Baseline",m]),"sustainable"])
    chi2,pc,_,_ = ss.chi2_contingency(ct)
    rows.append([m, round(pu,4), round(chi2,2), round(pc,4)])
print(pd.DataFrame(rows, columns=["vs Baseline","MWU p","chi2","chi2 p"]).to_string(index=False))

In [ ]:
# === 2c. Figure: distribution of extraction by mechanism ===================
fig, ax = plt.subplots(figsize=(11,6))
sns.violinplot(data=df, x="Mech", y="final_amount", order=ORDER,
               inner=None, cut=0, color="0.9", ax=ax)
sns.stripplot(data=df, x="Mech", y="final_amount", order=ORDER,
              jitter=.25, alpha=.5, size=4, ax=ax)
ax.axhline(50, ls="--", color="crimson", lw=1.5, label="sustainability threshold (≤50)")
ax.set(xlabel="", ylabel="Points extracted (final_amount)",
       title="Channel 1 — how much each institution lets people take")
ax.legend(loc="lower right", fontsize=11); plt.tight_layout(); plt.show()

## 3. Channel 2 — COORDINATION (behavioural compression)
The IA story isn't that people take dramatically less, it's that they *stop disagreeing*:
the distribution collapses onto 50. We test equality of dispersion directly.

In [ ]:
# === 3a. Dispersion tests ===================================================
print("Levene (median-centred / Brown-Forsythe) test of equal spread across conditions:")
W,p = ss.levene(*groups.values(), center="median")
print(f"  W={W:.2f}, p={p:.4f}\n")
print("Pairwise Brown-Forsythe vs Baseline:")
for m in ["IA","FAB","SDR","Selection"]:
    W,p = ss.levene(groups["Baseline"], groups[m], center="median")
    print(f"  Baseline vs {m:9s}: W={W:5.2f}, p={p:.4f}")

In [ ]:
# === 3b. Figure: ECDFs make the focal-point collapse obvious ===============
fig, ax = plt.subplots(figsize=(11,6))
for m in ORDER:
    x = np.sort(groups[m].values)
    y = np.arange(1, len(x)+1)/len(x)
    ax.step(x, y, where="post", lw=2.5, label=f"{m} (n={len(x)})")
ax.axvline(50, ls="--", color="crimson", lw=1.2)
ax.set(xlabel="Points extracted", ylabel="Cumulative share of participants",
       title="Channel 2 — IA compresses onto the 50-point focal step")
ax.legend(fontsize=12); plt.tight_layout(); plt.show()

## 4. Channel 3 — the SELF–OTHER and SELF–IDEAL gap  ⭐
This is the part your design uniquely captures.

**FAB** stores two continuous stages: `T2_1` = what you *request the predecessor* should take
(spoken as the next member), `T2_2` = what you actually *take as yourself*. `T2_3` records
which became final (1=request, 2=self, 3=unchanged). The gap `T2_2 − T2_1` is a
**self-serving hypocrisy index**.

**SDR** stores `T3_1` actual, `T3_2` ideal, `T3_3` ought, `T3_4` final, `T3_5` final selection.
The gap between *actual* and *ought* is a **moral discrepancy**; the question is whether the
final choice closes it.

In [ ]:
# === 4a. FAB: request vs self-take ==========================================
fab = df[df.Mech=="FAB"].copy()
fab[["T2_1","T2_2"]] = fab[["T2_1","T2_2"]].astype(float)
fab = fab.dropna(subset=["T2_1","T2_2"])
fab["gap"] = fab["T2_2"] - fab["T2_1"]          # self-take minus request-of-other
print("FAB stage means:  request(T2_1)=%.1f   self-take(T2_2)=%.1f   gap=%.1f"
      % (fab.T2_1.mean(), fab.T2_2.mean(), fab.gap.mean()))
w,p = ss.wilcoxon(fab["T2_1"], fab["T2_2"])
print(f"Wilcoxon signed-rank (request vs self-take): W={w:.1f}, p={p:.4g}")
print("\nWhich stage became final (T2_3): 1=request 2=self 3=unchanged")
print(fab["T2_3"].value_counts().sort_index().to_string())

In [ ]:
# === 4b. SDR: actual / ideal / ought / final ===============================
sdr = df[df.Mech=="SDR"].copy()
for c in ["T3_1","T3_2","T3_3","T3_4"]:
    sdr[c] = sdr[c].astype(float)
sdr = sdr.dropna(subset=["T3_1","T3_2","T3_3","T3_4"])
print("SDR self means:  actual(T3_1)=%.1f  ideal(T3_2)=%.1f  ought(T3_3)=%.1f  final(T3_4)=%.1f"
      % (sdr.T3_1.mean(), sdr.T3_2.mean(), sdr.T3_3.mean(), sdr.T3_4.mean()))
chi2,p = ss.friedmanchisquare(sdr.T3_1, sdr.T3_2, sdr.T3_3)
print(f"Friedman across actual/ideal/ought:  chi2={chi2:.2f}, p={p:.4g}")
# Does the moral gap predict closing the gap?
sdr["moral_gap"]   = sdr["T3_1"] - sdr["T3_3"]          # actual minus ought (how far short)
sdr["closed"]      = sdr["T3_1"] - sdr["T3_4"]          # how much the final moved down
r,pr = ss.spearmanr(sdr["moral_gap"], sdr["closed"])
print(f"\nSpearman(moral gap, downward revision in final): rho={r:.2f}, p={pr:.4g}")
print("  → positive rho = people with a bigger actual–ought gap revise further toward the ought")

In [ ]:
# === 4c. Figure: the discrepancy story ======================================
fig, axes = plt.subplots(1, 2, figsize=(14,6))

# FAB paired slopes
for _, row in fab.iterrows():
    axes[0].plot([0,1], [row.T2_1, row.T2_2], color="steelblue", alpha=.25)
axes[0].plot([0,1], [fab.T2_1.mean(), fab.T2_2.mean()], color="black", lw=3, marker="o")
axes[0].set(xticks=[0,1], xticklabels=["request\n(of predecessor)","self-take"],
            ylabel="Points", title="FAB: the hypocrisy gap")
axes[0].axhline(50, ls="--", color="crimson", lw=1)

# SDR selves
sdr_long = sdr[["T3_1","T3_2","T3_3","T3_4"]].mean()
axes[1].bar(["actual","ideal","ought","final"], sdr_long.values,
            color=["#c44","#48a","#4a4","#888"])
axes[1].axhline(50, ls="--", color="crimson", lw=1)
axes[1].set(ylabel="Mean points", title="SDR: actual vs ought vs final")
plt.tight_layout(); plt.show()

## 5. Social value orientation (SVO) as a moderator
`svo1`/`svo2` are the **summed** "You"/"Other" allocations over the six Murphy-et-al. slider
items, so the per-item means are `svo1/6` and `svo2/6`. The SVO angle is
`arctan((Ao−50)/(As−50))`; prosocial if angle > 22.45°.

> ⚠️ Theoretical max of `svo1` is ~555 (max-You option on every item). A few rows exceed that
> (data-entry slips) and are flagged below — decide whether to drop them.

In [ ]:
# === 5. SVO construction + interaction with mechanism ======================
df["As"] = df["svo1"]/6.0
df["Ao"] = df["svo2"]/6.0
df["svo_angle"] = np.degrees(np.arctan2(df["Ao"]-50, df["As"]-50))
df["prosocial"] = (df["svo_angle"] > 22.45).astype(int)

bad = df["svo1"] > 560
print(f"Suspect SVO rows (svo1>560, likely entry errors): {bad.sum()}")
print("\nProsocial share and sustainability by mechanism:")
print(df.groupby("Mech").agg(prosocial_share=("prosocial","mean"),
                             pct_sustainable=("sustainable", lambda x:100*x.mean()))
        .reindex(ORDER).round(3).to_string())

print("\nSustainable share by mechanism × SVO type (does IA rescue proself players?):")
print(pd.crosstab(df["Mech"], df["prosocial"], values=df["sustainable"],
                  aggfunc="mean").reindex(ORDER).round(2).to_string())

## 6. Regression — putting it together
Two complementary models, paralleling Table 5 of the 2019 paper:

* **Logit** on the sustainable dummy (clean, gives marginal effects).
* **Tobit** on `final_amount`, double-censored at 0 and 100 (correct for the top/bottom pile-ups).

Baseline is the reference category; controls: prosocial, sex, age, prior-experiment.

In [ ]:
# === 6a. Logit on sustainable dummy ========================================
d = df.dropna(subset=["final_amount","prosocial","sex","age","experiment"]).copy()
D = pd.get_dummies(d["Mech"], prefix="M")
for c in ["M_IA","M_FAB","M_SDR","M_Selection"]:
    if c not in D: D[c]=0
X = pd.concat([D[["M_IA","M_FAB","M_SDR","M_Selection"]],
               d[["prosocial","sex","age","experiment"]]], axis=1).astype(float)
X = sm.add_constant(X)
logit = sm.Logit(d["sustainable"].astype(float), X).fit(disp=0)
print(logit.summary2().tables[1].round(3))
print("\nAverage marginal effects:")
print(logit.get_margeff().summary_frame().round(3))

In [ ]:
# === 6b. Tobit (double-censored at 0 and 100) ==============================
from scipy.optimize import minimize
from scipy.stats import norm
y = d["final_amount"].astype(float).values
Xm = X.values.astype(float)
lo, hi = 0.0, 100.0

def negll(theta):
    b, ls = theta[:-1], theta[-1]
    s = np.exp(ls); xb = Xm @ b; z=(y-xb)/s
    ll = np.where(y<=lo, norm.logcdf((lo-xb)/s),
         np.where(y>=hi, norm.logsf((hi-xb)/s),
                  norm.logpdf(z)-ls))
    return -ll.sum()

init = np.r_[np.linalg.lstsq(Xm, y, rcond=None)[0], np.log(y.std())]
res = minimize(negll, init, method="Nelder-Mead", options={"maxiter":20000,"xatol":1e-4,"fatol":1e-4})
names = list(X.columns)+["log_sigma"]
print("Tobit coefficients (final_amount, censored 0–100):")
print(pd.Series(res.x, index=names).round(3).to_string())
print("\nReading: negative IA coefficient = IA lowers extraction relative to Baseline.")

## 7. Endogenous selection — preference vs effectiveness
When free to choose (`Treatment==5`), what do people pick, and do choosers do better than
the randomly-assigned? `mechanism`: 1=IA, 2=FAB, 3=SDR.

In [ ]:
# === 7. Selection analysis =================================================
sel = df[df.Mech=="Selection"].copy()
sel["chosen"] = sel["mechanism"].map({1:"IA",2:"FAB",3:"SDR"})
print("What people choose when free to choose:")
print(sel["chosen"].value_counts().to_string())

print("\nSustainable share: self-selected vs randomly-assigned to the SAME mechanism")
for m in ["IA","FAB","SDR"]:
    a = df.loc[df.Mech==m, "sustainable"].mean()
    b = sel.loc[sel.chosen==m, "sustainable"].mean()
    print(f"  {m:4s}  assigned={100*a:5.1f}%   chosen={100*b:5.1f}%")
# overall level test
u,p = ss.mannwhitneyu(df.loc[df.Mech=="Baseline","final_amount"],
                      sel["final_amount"], alternative="two-sided")
print(f"\nSelection vs Baseline extraction (MWU): p={p:.4f}")

## 8. Where to take it next
* **Sequence / adverse-history effects (H2).** The raw `Generation_1`/`Generation_2`
  workbooks carry the order of members within each sequence and the running account.
  Merge a within-sequence position index in, then test whether each mechanism *holds up*
  as the resource gets depleted — that is the "when sustainability is endangered" test.
* **Unit of analysis.** Decide member-level vs sequence-level before writing up; the
  presentation's N's (30, 23, …) are sequences, the `.dta` rows are members.
* **Multiple comparisons.** Apply Holm/BH correction to the pairwise tests for the paper.
